# Exploration — Telco Customer Churn

Premier contact avec la donnée du fil rouge. Objectif du Chapitre 3 : savoir
ce qu'on a sous la main, et fixer la **baseline métier** que tout modèle devra battre.

Aucune rigueur ici, c'est assumé : tout tient dans un notebook, les chemins sont
relatifs, rien n'est testé. Le Chapitre 4 nettoie exactement ça.

In [ ]:
import pandas as pd

df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)
print(df["Churn"].value_counts(normalize=True))

Sortie attendue :

```
(7043, 21)
Churn
No     0.7346
Yes    0.2654
```

7043 clients, 21 colonnes. 26,5 % sont partis — le jeu est **déséquilibré**.

## La baseline métier

Un modèle qui répondrait toujours « ce client reste » aurait mécaniquement
73,5 % de bonnes réponses. C'est le seuil minimal : en dessous, un modèle de
machine learning ne sert littéralement à rien.

Et même au-dessus, l'accuracy seule ment : ce modèle stupide ne détecte
**aucun** des clients qui partent — précisément ceux que l'entreprise veut voir venir.

In [ ]:
baseline_accuracy = (df["Churn"] == "No").mean()
print(f"Accuracy d'un modele qui predit toujours 'No' : {baseline_accuracy:.1%}")
print(f"Recall de ce meme modele sur la classe 'Yes'  : 0.0%  <- le probleme")

## Un coup d'oeil aux colonnes

In [ ]:
df.info()

`TotalCharges` sort en `object` (texte), pas en `float`. Onze clients tout neufs
(`tenure = 0`) y ont une chaîne vide au lieu d'un nombre. C'est le genre de détail
qui fait planter l'entraînement quelques cellules plus bas — autant le repérer maintenant.

In [ ]:
blancs = (df["TotalCharges"].str.strip() == "").sum()
print(f"Lignes avec TotalCharges vide : {blancs}")
print(df.loc[df["TotalCharges"].str.strip() == "", ["customerID", "tenure", "MonthlyCharges"]].head())

## Où se cache le churn ?

Le taux de désabonnement par type de contrat : le signal le plus net du dataset.

In [ ]:
churn_par_contrat = (
    df.assign(churn=(df["Churn"] == "Yes"))
      .groupby("Contract")["churn"]
      .mean()
      .sort_values(ascending=False)
)
print((churn_par_contrat * 100).round(1))

Sortie attendue :

```
Contract
Month-to-month    42.7
One year          11.3
Two year           2.8
```

Un client en contrat mensuel part quinze fois plus souvent qu'un client engagé
deux ans. Un modèle a donc bien quelque chose à apprendre ici — c'est ce que
le modèle entraîné en fin de notebook confirmera, chiffres à l'appui.

## Le piège des onze lignes

`TotalCharges` doit devenir numérique avant tout entraînement. Onze clients
d'ancienneté nulle y ont un espace vide : sans ces deux lignes, l'entraînement
plante sur `ValueError: could not convert string to float: ' '`.

In [ ]:
# to_numeric convertit ce qui peut l'etre et met NaN sur le reste ;
# fillna(0) traduit l'hypothese metier : un client arrive ce mois-ci n'a
# encore rien facture.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
print(df["TotalCharges"].dtype)

## Le churn par ancienneté

Deuxième signal le plus net du dataset, après le type de contrat.

In [ ]:
tranches = pd.cut(df["tenure"], [-1, 12, 24, 48, 100],
                  labels=["0-12", "13-24", "25-48", "49+"])
print((pd.crosstab(tranches, df["Churn"], normalize="index") * 100).round(1))

Sortie attendue :

```
Churn     No   Yes
tenure
0-12    52.6  47.4
13-24   71.3  28.7
25-48   79.6  20.4
49+     90.5   9.5
```

Un client de moins d'un an part une fois sur deux ; passé quatre ans, une fois
sur dix. La première année est le vrai champ de bataille.

## Séparer les features du label

`customerID` sort : c'est un identifiant unique, chaque valeur n'apparaît
qu'une fois, donc il ne porte aucun signal généralisable. `Churn` sort aussi
de `X` — c'est la réponse, pas une information d'entrée.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["customerID", "Churn"])
y = (df["Churn"] == "Yes").astype(int)   # Yes/No -> 1/0, format attendu par sklearn

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

Sortie attendue :

```
(5634, 19) (1409, 19)
```

`random_state=42` fige le tirage aléatoire. Sans lui, deux exécutions donnent
deux découpages différents, donc deux scores incomparables. C'est le premier
geste de reproductibilité du projet — le Chapitre 7 en fera un système.

## Le baseline, noté avant d'entraîner quoi que ce soit

Le noter après, c'est se laisser la possibilité de trouver 79 % impressionnant.

In [ ]:
from sklearn.metrics import accuracy_score

y_baseline = [0] * len(y_test)   # « aucun client ne part »
print("Baseline :", round(accuracy_score(y_test, y_baseline), 4))

Sortie attendue :

```
Baseline : 0.7353
```

## Le premier vrai modèle

`OneHotEncoder` transforme chaque colonne texte en colonnes binaires — un
modèle ne sait pas lire `"Month-to-month"`. Le `Pipeline` enchaîne cet encodage
et le modèle en **un seul objet** : un `.fit()`, un `.predict()`, et surtout un
seul fichier à sauvegarder. Sans lui, il faudrait se souvenir « quelle colonne
encoder comment » au moment de prédire, et la moindre divergence produirait des
prédictions fausses, silencieusement.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

colonnes_texte = X.select_dtypes(include="object").columns.tolist()

preparation = ColumnTransformer(
    # handle_unknown="ignore" evite un plantage si une valeur inedite
    # apparait un jour en production.
    [("cat", OneHotEncoder(handle_unknown="ignore"), colonnes_texte)],
    # remainder="passthrough" laisse passer les colonnes deja numeriques.
    remainder="passthrough",
)

pipeline = Pipeline([
    ("prep", preparation),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42)),
])

pipeline.fit(X_train, y_train)
print("Entrainement termine")

## Mesurer honnêtement

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = pipeline.predict(X_test)

print("Accuracy test  :", round(accuracy_score(y_test, y_pred), 4))
print("Accuracy train :", round(pipeline.score(X_train, y_train), 4))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=2))

Sortie attendue :

```
Accuracy test  : 0.7913
Accuracy train : 0.9986
[[942  94]
 [200 173]]
              precision    recall  f1-score   support

           0       0.82      0.91      0.87      1036
           1       0.65      0.46      0.54       373

    accuracy                           0.79      1409
   macro avg       0.74      0.69      0.70      1409
weighted avg       0.78      0.79      0.78      1409
```

Trois lectures, dans cet ordre :

- **0,7913 en test contre 0,7353 de baseline** : le modèle apporte 5,6 points.
  Réel, mais modeste.
- **0,9986 en train contre 0,7913 en test** : surapprentissage caractérisé, la
  forêt a mémorisé le jeu d'entraînement.
- **Recall de 0,46 sur la classe 1** : le modèle voit 173 départs sur 373. Il en
  rate 200.

Le chiffre qui compte pour le métier est le dernier. Une accuracy de 79 % avait
l'air rassurante ; le recall dit qu'un client à risque sur deux passe entre les
mailles.

Ce modèle n'est pas bon. **Et c'est le sujet du livre** : un modèle médiocre
mais traçable, déployé et surveillé bat un modèle brillant coincé dans un
notebook.

## Sauvegarder le modèle

In [ ]:
import joblib

joblib.dump(pipeline, "model.pkl")
print("modele sauvegarde")

## Ce qui ne va pas dans ce notebook

- le chemin du CSV est en dur et relatif : il casse dès qu'on lance depuis ailleurs
- `model.pkl` atterrit à la racine, à côté du notebook, sans version ni métadonnée
- aucune trace des hyperparamètres utilisés : `n_estimators=200` n'existe que dans
  cette cellule
- rien n'est réutilisable : ni test, ni API, ni pipeline ne peut appeler ces cellules
- l'ordre d'exécution des cellules n'est garanti par rien

C'est l'état dans lequel se trouvent la plupart des projets ML du monde.
**Le Chapitre 4 s'attaque à ce désordre.**